### Supplement: 2-Structured Outputs with Pydantic Deep Dive

This supplement notebook breaks down `2-structured.py` cell by cell. It covers **Structured Outputs** using Pydantic and the OpenAI Python SDK:

- **Section 2 — What is Pydantic and `BaseModel`?**: Defining data schemas with strict types. **Schema Generation**: How `CalendarEvent` is converted to a JSON Schema under the hood, and who uses it.
- **Section 3 — `create` vs. `parse`**: Understanding `client.chat.completions.parse(...)`.
- **Section 4 — `message.content` vs. `message.parsed`**:
   - `message.content`: The raw JSON string returned by the LLM.
   - `message.parsed`: The deserialized, validated Pydantic object!
   - plus a tree of the whole response, showing where `.parsed` sits.
- **Section 5 — Accessing Typed Attributes & Converting to Python Dicts**: Dot-notation vs. `.model_dump()`.
- **Section 6 — The complete response object**: the whole `completion` as a dict, and the Pydantic warning it prints.
- **Section 7 — Cheat Sheet**: `model_json_schema()` vs. `model_dump()` vs. `json.loads()` vs. `.parsed`.

---

##### Architectural Flow:
```
1. Define Pydantic Schema:
   class CalendarEvent(BaseModel):
       name: str
       date: str
       participants: list[str]
            │
            ▼
2. SDK calls .model_json_schema(), tightens it for strict mode,
   and puts it in the request body sent to OpenAI
            │
            ▼
3. OpenAI API uses Constrained Sampling (the emitted JSON is
   guaranteed to match the schema — unless the model refuses,
   or generation is cut off by a token limit)
            │
            ▼
4. Response arrives back at SDK:
   ├── message.content -> raw JSON string: '{"name": "...", "date": "...", ...}'
   └── message.parsed  -> CalendarEvent(name='...', date='...', participants=[...])
```

> **Note on `.beta.`**: older tutorials call `client.beta.chat.completions.parse(...)`. Structured-output parsing has since graduated out of beta, so this notebook uses `client.chat.completions.parse(...)` — matching `2-structured.py`. In the installed SDK (openai 3.14.1), `client.beta.chat.completions` still exists, but it is now simply another name for `client.chat.completions`: the same class, with the same `parse` function. So older code keeps working and behaves identically, including the `ParsedChatCompletion[TypeVar]` type name you'll see in Section 3. If an older tutorial shows `ParsedChatCompletion[CalendarEvent]`, it was written against an older SDK version. Prefer the non-`beta` path in anything new.

#### 1. Imports and Environment Setup

Notice we import:
- `OpenAI`: The API client class.
- `BaseModel`, `Field`: From `pydantic`. `BaseModel` is the base class for defining data contracts and schemas.

In [2]:
import os
import json
from dotenv import load_dotenv, find_dotenv
from openai import OpenAI
from pydantic import BaseModel, Field

# Load API key
load_dotenv(find_dotenv(usecwd=True))
if not os.getenv("OPENAI_API_KEY"):
    load_dotenv(r"C:\Users\ashut\ML_Practice\LLM_Learning_Sandbox\.env")

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
print("Client initialized successfully.")
print("Pydantic BaseModel class:", BaseModel)


Client initialized successfully.
Pydantic BaseModel class: <class 'pydantic.main.BaseModel'>


#### 2. Defining the Schema Class: `CalendarEvent(BaseModel)`

##### What is `BaseModel`?
`BaseModel` is Pydantic's core class. When you subclass `BaseModel`:
- It parses and validates data according to type hints (`str`, `list[str]`, etc.).
- It can automatically export a standard JSON Schema via `.model_json_schema()`.
- It allows serialization back to dicts via `.model_dump()`.

Below we call `.model_json_schema()` **purely for inspection**, so you can see the shape the SDK will derive from your class. Note that nothing in this notebook sends that `schema` variable anywhere — the SDK regenerates it internally when you pass `response_format=CalendarEvent` in Section 3. The next cell explains exactly who consumes it.

In [3]:
class CalendarEvent(BaseModel):
    name: str = Field(description="The name or title of the event")
    date: str = Field(description="The date or day of the event")
    participants: list[str] = Field(description="List of attendee names")

print("Class name:", CalendarEvent.__name__)
print("Inherits from:", [b.__name__ for b in CalendarEvent.__bases__])
print("Declared fields:", list(CalendarEvent.model_fields.keys()))

print("\n--- JSON Schema generated by Pydantic (Sent to OpenAI API under the hood) ---")
schema = CalendarEvent.model_json_schema()
print(json.dumps(schema, indent=2))


Class name: CalendarEvent
Inherits from: ['BaseModel']
Declared fields: ['name', 'date', 'participants']

--- JSON Schema generated by Pydantic (Sent to OpenAI API under the hood) ---
{
  "properties": {
    "name": {
      "description": "The name or title of the event",
      "title": "Name",
      "type": "string"
    },
    "date": {
      "description": "The date or day of the event",
      "title": "Date",
      "type": "string"
    },
    "participants": {
      "description": "List of attendee names",
      "items": {
        "type": "string"
      },
      "title": "Participants",
      "type": "array"
    }
  },
  "required": [
    "name",
    "date",
    "participants"
  ],
  "title": "CalendarEvent",
  "type": "object"
}


##### Wait — who actually *uses* this JSON Schema, and for what?

This is the single most confusing part of structured outputs, so let's be precise.

**You never call `.model_json_schema()` yourself in normal usage.** The cell above is a teaching X-ray. The schema's real consumer is **OpenAI's inference server**, not your Python code.

Here is the actual chain of custody:

1. You pass the **class itself** (not a schema) to the SDK: `response_format=CalendarEvent`.
2. **The SDK** calls `CalendarEvent.model_json_schema()` for you. The call starts in `openai/lib/_parsing/_completions.py` (`type_to_response_format_param`), which hands your class to `to_strict_json_schema` in `openai/lib/_pydantic.py`. That is where `model_json_schema()` actually runs.
3. **The SDK tightens it for strict mode**: it recursively injects `"additionalProperties": false` into every object and wraps the result with a `name` and `strict: true`.
4. That payload goes over the wire in the request body.
5. **OpenAI's server** compiles the schema into a grammar and uses it for **constrained decoding** — at each step, tokens that would break the schema are masked out of the sampling distribution. The model is *mechanically unable* to emit a wrong field name, a wrong type, or a missing required field.

So the answer to *"where is this JSON used, by whom, for what?"* is: **by OpenAI's token sampler, to make invalid output impossible.** It is a contract shipped to the model, not data for your program.

##### Important: the raw Pydantic schema is *not* byte-for-byte what gets sent

The printout above is Pydantic's generic export. Compare it to what the SDK actually transmits:

```python
from openai.lib._parsing._completions import type_to_response_format_param
print(json.dumps(type_to_response_format_param(CalendarEvent), indent=2))
```

```jsonc
{
  "type": "json_schema",
  "json_schema": {
    "schema": {
      "properties": { /* ...same as above... */ },
      "required": ["name", "date", "participants"],
      "title": "CalendarEvent",
      "type": "object",
      "additionalProperties": false   // <-- ADDED by the SDK for strict mode
    },
    "name": "CalendarEvent",          // <-- ADDED
    "strict": true                    // <-- ADDED
  }
}
```

Two practical consequences:
- The `description=` text you wrote in each `Field(...)` **does** survive into the schema, so it reaches the model and acts as a per-field prompt. Descriptive `Field` descriptions genuinely improve extraction quality.
- Strict mode forbids optional/extra keys. Every field is required, which is why you model "maybe missing" as `Optional[str]` (i.e. `str | None`) rather than by omitting the field.

#### 3. The API Call: `client.chat.completions.parse(...)`

##### Why `parse(...)` instead of `create(...)`?
- **`create(...)`**: You get back a standard `ChatCompletion`. `message.content` is just a `str`, and `message.parsed` does not exist. To get an object you must deserialize it yourself:
  ```python
  data = json.loads(completion.choices[0].message.content)  # -> plain dict
  event = CalendarEvent(**data)                             # -> validate by hand
  ```
- **`parse(...)`**: A high-level helper in the OpenAI SDK that:
  1. Converts your Pydantic class to a JSON Schema and sends it with `strict: true`.
  2. The model generates strictly conforming JSON tokens via constrained decoding.
  3. The SDK automatically validates the JSON and instantiates your `CalendarEvent` class!
  4. The instantiated object is placed in `completion.choices[0].message.parsed`.

> **A precise distinction.** `create()` is not inherently "unsafe JSON" — it also accepts `response_format={"type": "json_schema", ...}` with strict mode, giving the same generation guarantee. The catch is that you must hand-write that schema dict and do your own `json.loads()` + validation. So `parse()` is not buying you *reliability the API otherwise lacks*; it is buying you **the Pydantic-class-to-schema conversion on the way out, and the deserialization on the way back.**
>
> The older "JSON mode" (`response_format={"type": "json_object"}`) is the genuinely weaker option: it guarantees only *syntactically valid* JSON, with no control over which fields appear. That is the case where "hope the model didn't invent fields" actually applies.

##### Don't be thrown by `ParsedChatCompletion[TypeVar]` in the output below

- **What it is:** `ParsedChatCompletion` is a *generic* class: the square brackets are a slot meant to hold the type of `.parsed`, so you'd expect `ParsedChatCompletion[CalendarEvent]`. `parse()` builds the object without filling that slot in, so Python prints the unfilled placeholder, `TypeVar`. The `.beta.` path is now the same function, so it prints the same thing.
- **The key point:** it is *mostly* cosmetic. Your data is unaffected: Section 4 confirms that `message.parsed` is a genuine `CalendarEvent` and `isinstance(message.parsed, CalendarEvent)` is `True`. Static type checkers still infer the correct type in your editor.
- **The one visible side effect:** because the slot is unfilled, Pydantic doesn't know that `.parsed` holds a `CalendarEvent`. So `completion.model_dump()` in Section 6 prints a harmless `UserWarning`. Section 6 explains exactly why.

In [4]:
prompt_messages = [
    {"role": "system", "content": "Extract the event information."},
    {
        "role": "user",
        "content": "Alice and Bob are going to a science fair on Friday.",
    },
]

completion = client.chat.completions.parse(
    model="gpt-5-nano",
    messages=prompt_messages,
    response_format=CalendarEvent,
)

print("Parsed completion call successful!")
print("Completion object type:", type(completion))
print("Choices length:", len(completion.choices))

Parsed completion call successful!
Completion object type: <class 'openai.types.chat.parsed_chat_completion.ParsedChatCompletion[TypeVar]'>
Choices length: 1


#### 4. Comparing `message.content` vs. `message.parsed`

This is the most critical distinction in Structured Outputs:
1. **`message.content`**: Contains the **raw JSON string** sent across the wire by the LLM.
2. **`message.parsed`**: Contains the **instantiated Python Pydantic object** (`CalendarEvent`).

Both are populated on a successful call — `parsed` is simply the SDK having already done `json.loads()` + Pydantic validation on `content` for you.

The cell also prints **`message.refusal`**, which is the escape hatch in the schema guarantee. If the model declines the request on safety grounds, it returns a plain-text refusal *instead of* schema-conforming JSON: `refusal` holds that text, and `parsed` is `None`. In production this is what you branch on:

```python
if message.refusal:
    handle_refusal(message.refusal)
else:
    event = message.parsed
```

Let's inspect all three with `type()` and `repr()`:

In [5]:
message = completion.choices[0].message

print("=== 1. message.content (Raw Wire JSON) ===")
print("Type:", type(message.content))
print("Raw string value:", repr(message.content))

print("\n=== 2. message.parsed (Deserialized Pydantic Object) ===")
print("Type:", type(message.parsed))
print("Is instance of CalendarEvent?:", isinstance(message.parsed, CalendarEvent))
print("Object representation:", repr(message.parsed))

print("\n=== 3. message.refusal ===")
print("Refusal status:", message.refusal)


=== 1. message.content (Raw Wire JSON) ===
Type: <class 'str'>
Raw string value: '{"name":"Science Fair","date":"Friday","participants":["Alice","Bob"]}'

=== 2. message.parsed (Deserialized Pydantic Object) ===
Type: <class '__main__.CalendarEvent'>
Is instance of CalendarEvent?: True
Object representation: CalendarEvent(name='Science Fair', date='Friday', participants=['Alice', 'Bob'])

=== 3. message.refusal ===
Refusal status: None


##### Visualizing the Nested Hierarchy: where `message.parsed` sits

This is the same walk as the `.content` tree in `Supplement_1-basic.ipynb` (Section 5), redrawn for a `parse()` response. `.parsed` sits next to `.content`, and it has children of its own: the fields you declared in `CalendarEvent`.

<pre style="font-size: 13.5px; line-height: 1.38; font-family: Consolas, 'Courier New', monospace; padding: 8px 12px; border-radius: 5px; white-space: pre; overflow-x: auto;">completion                                            <span style="font-size: 0.8em;">&lt;class 'openai.types.chat.parsed_chat_completion.ParsedChatCompletion[TypeVar]'&gt;</span>
  │
  ├── .choices                                        <span style="font-size: 0.8em;">&lt;class 'list'&gt;</span>
  │     │
  │     └── [0]                                       <span style="font-size: 0.8em;">&lt;class 'openai.types.chat.parsed_chat_completion.ParsedChoice[TypeVar]'&gt;</span>
  │           │
  │           ├── .index: 0                           <span style="font-size: 0.8em;">&lt;class 'int'&gt;</span>
  │           ├── .finish_reason: 'stop'              <span style="font-size: 0.8em;">&lt;class 'str'&gt;</span>
  │           └── .message                            <span style="font-size: 0.8em;">&lt;class 'openai.types.chat.parsed_chat_completion.ParsedChatCompletionMessage[TypeVar]'&gt;</span>
  │                 │
  │                 ├── .role: 'assistant'            <span style="font-size: 0.8em;">&lt;class 'str'&gt;</span>
  │                 ├── .content: '{"name":...}'      <span style="font-size: 0.8em;">&lt;class 'str'&gt; (the raw JSON text the model generated)</span>
  │                 ├── .refusal: None                <span style="font-size: 0.8em;">&lt;class 'NoneType'&gt;</span>
  │                 └── .parsed                       <span style="font-size: 0.8em;">&lt;class '__main__.CalendarEvent'&gt; (YOUR class, built from .content by the SDK)</span>
  │                       │
  │                       ├── .name: 'Science Fair'   <span style="font-size: 0.8em;">&lt;class 'str'&gt;</span>
  │                       ├── .date: 'Friday'         <span style="font-size: 0.8em;">&lt;class 'str'&gt;</span>
  │                       └── .participants           <span style="font-size: 0.8em;">&lt;class 'list'&gt;</span>
  │                             │
  │                             ├── [0]: 'Alice'      <span style="font-size: 0.8em;">&lt;class 'str'&gt;</span>
  │                             └── [1]: 'Bob'        <span style="font-size: 0.8em;">&lt;class 'str'&gt;</span></pre>

What's different from the `create()` tree:
- **Each class on the path gets a `Parsed` prefix.** `ChatCompletion` becomes `ParsedChatCompletion`, `Choice` becomes `ParsedChoice`, and `ChatCompletionMessage` becomes `ParsedChatCompletionMessage`. Each one is a subclass of the original. `ParsedChatCompletionMessage` is the class that adds the `parsed` field. The other two change their child's type so that the path leads down to it.
- **Everything above `.parsed` uses OpenAI's classes. `.parsed` itself is your `CalendarEvent`.** Its children are exactly the three fields you declared.
- **The two parts are built differently.** The SDK builds the outer layers with the lenient `construct_type`, which checks no types (see `Supplement_1` Section 4b). It builds `.parsed` with `CalendarEvent.model_validate_json(message.content)`, which runs full Pydantic validation.

#### 5. Accessing Typed Attributes & Converting to Python Dict

Because `event` is a `CalendarEvent` object:
- You get typed attribute access (dot-notation) with IDE autocompletion: `event.name`, `event.date`, `event.participants`.
- `event.participants` is a genuine Python `list` of strings!
- You can convert the object to a standard Python dictionary using `event.model_dump()`.

In [6]:
event: CalendarEvent = message.parsed

print("--- Accessing Typed Attributes ---")
print(f"event.name:         {event.name} (type: {type(event.name)})")
print(f"event.date:         {event.date} (type: {type(event.date)})")
print(f"event.participants: {event.participants} (type: {type(event.participants)})")
print(f"First participant:  {event.participants[0]} (type: {type(event.participants[0])})")

print("\n--- Converting to Standard Python Dictionary (.model_dump()) ---")
event_dict = event.model_dump()
print("Type of event_dict:", type(event_dict))
print("Dictionary content:", event_dict)


--- Accessing Typed Attributes ---
event.name:         Science Fair (type: <class 'str'>)
event.date:         Friday (type: <class 'str'>)
event.participants: ['Alice', 'Bob'] (type: <class 'list'>)
First participant:  Alice (type: <class 'str'>)

--- Converting to Standard Python Dictionary (.model_dump()) ---
Type of event_dict: <class 'dict'>
Dictionary content: {'name': 'Science Fair', 'date': 'Friday', 'participants': ['Alice', 'Bob']}


#### 6. Visualizing the Complete Structured Response Object

- **What the next cell does:** converts the whole `completion` into a dict with `completion.model_dump()`, stores it as `full_dict`, and prints it with `json.dumps(..., indent=2)`.
- **The key point:** `message` holds the same data twice, side by side: `content` (the JSON text the model wrote) and `parsed` (your `CalendarEvent`, now as a nested dict).
- **Also look at:** `usage.completion_tokens_details.reasoning_tokens`. Most of the completion tokens are hidden reasoning (see `Supplement_1-basic.ipynb`, Section 7).

Let's dump the entire `completion` object to inspect everything OpenAI returned, including token usage and choice metadata:

##### About the `UserWarning` printed under the output

After the dict, the cell prints a warning, `PydanticSerializationUnexpectedValue`, saying it expected `none` for the field `parsed`.

- **What it is:** Pydantic complaining that `parsed` holds a value of a type it didn't expect.
- **The key point:** it is harmless. The dict above is complete and correct, `parsed` included. Pydantic warns, then converts the value anyway.

Why it happens: it comes from the `[TypeVar]` slot described in Section 3.

1. `parse()` builds the response as `ParsedChatCompletion[ResponseFormatT]`, where `ResponseFormatT` is the unfilled placeholder.
2. The SDK declares that placeholder with a default of `None`, meaning "nothing to parse" (`openai/lib/_parsing/_completions.py`: `ResponseFormatT = TypeVar("ResponseFormatT", default=None)`).
3. Because the slot is never filled in, Pydantic falls back to that default and believes `parsed` should always be `None`.
4. During `model_dump()` it finds a `CalendarEvent` there instead, so it prints the warning, then serializes it anyway.

In [ ]:
full_dict = completion.model_dump()

print("Full Completion Dictionary:")
print(json.dumps(full_dict, indent=2, warnings=False))


Full Completion Dictionary:
{
  "id": "chatcmpl-EPCKDPLr9Le2JXNU0xgieEeXtUYED",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": "{\"name\":\"Science Fair\",\"date\":\"Friday\",\"participants\":[\"Alice\",\"Bob\"]}",
        "refusal": null,
        "role": "assistant",
        "annotations": [],
        "audio": null,
        "function_call": null,
        "tool_calls": null,
        "parsed": {
          "name": "Science Fair",
          "date": "Friday",
          "participants": [
            "Alice",
            "Bob"
          ]
        }
      }
    }
  ],
  "created": 1789674285,
  "model": "gpt-5-nano-2025-08-07",
  "object": "chat.completion",
  "metadata": null,
  "moderation": null,
  "service_tier": "default",
  "system_fingerprint": null,
  "usage": {
    "completion_tokens": 414,
    "prompt_tokens": 117,
    "total_tokens": 531,
    "completion_tokens_details": {
      "accepted_predictio

c:\Users\ashut\anaconda3\envs\General_env\Lib\site-packages\pydantic\main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=CalendarEvent(name='Scien...ipants=['Alice', 'Bob']), input_type=CalendarEvent])
  return self.__pydantic_serializer__.to_python(


---

#### 7. Cheat Sheet: `model_json_schema()` vs. `model_dump()` vs. `json.loads()` vs. `.parsed`

These names blur together because they all involve "JSON" — but they move in **different directions** and belong to **different libraries**. Sort them by *what goes in* and *what comes out*.

##### The one distinction that unlocks the rest

- `model_json_schema()` describes the **shape** — it operates on the **class** and produces a *description of a type*. It contains no data. It travels **outward to the model**.
- `model_dump()` / `model_dump_json()` carry the **data** — they operate on an **instance** and produce *values*. They travel **outward to your code, a file, or another service**.

> A schema is the mould; a dump is the casting. `CalendarEvent.model_json_schema()` works without any event ever existing, whereas `event.model_dump()` needs a real `event`.

##### The same idea, in plain words

Think of `CalendarEvent` as a **blank paper form** with three boxes to fill in: *Name*, *Date* and *Participants*.

- The **class** (`CalendarEvent`) is the blank form. An **instance** (`event`) is one copy of that form with the boxes filled in.
- **`model_json_schema()` describes the blank form.** A blank form still has things printed on it: a label next to each box, a short instruction under it, and an idea of what kind of answer fits (some text, a list of names). The schema describes exactly those printed parts. It can't mention "Science Fair", because a blank form has no answers written on it. That's what "it contains no data" means: no *answers*, even though it does contain some text.
- **`model_dump()` reads the answers off one filled-in form.** It says "name is Science Fair". There has to be a filled-in form to read from, which is why it works on an *instance* and not on the class.

##### Seeing it in code

The five steps below answer one question: **what does `model_json_schema()` give you, and how is that different from what `model_dump()` gives you?**

All five use the `CalendarEvent` class from Section 2:

```python
class CalendarEvent(BaseModel):
    name: str = Field(description="The name or title of the event")
    date: str = Field(description="The date or day of the event")
    participants: list[str] = Field(description="List of attendee names")
```

---

**Step 1: `model_json_schema()` describes the class**

- **What it is:** `model_json_schema()` is a Pydantic method that you call on the **class**, `CalendarEvent`, not on an event. It returns a Python `dict` describing the class: which fields it has, what type each one is, and the description you wrote for each one.
- **The key point:** it describes the fields but contains **no values**. You won't find "Science Fair", "Friday" or "Alice" anywhere in it. It can't contain them, because no event has been created yet. All it knows is what the class definition says.

We call it and store the result in a variable named `schema`:

```python
schema = CalendarEvent.model_json_schema()
print(schema)
```

Output (a dict prints on one long line):

```
{'properties': {'name': {'description': 'The name or title of the event', 'title': 'Name', 'type': 'string'}, 'date': {'description': 'The date or day of the event', 'title': 'Date', 'type': 'string'}, 'participants': {'description': 'List of attendee names', 'items': {'type': 'string'}, 'title': 'Participants', 'type': 'array'}}, 'required': ['name', 'date', 'participants'], 'title': 'CalendarEvent', 'type': 'object'}
```

That is hard to read, so here is the same dict spread over several lines. `json.dumps(schema, indent=2)`, from Python's built-in `json` module, turns the dict into text with one item per line. The content is identical. The quotes become double quotes only because `json.dumps` produces JSON text.

```json
{
  "properties": {
    "name": {
      "description": "The name or title of the event",
      "title": "Name",
      "type": "string"
    },
    "date": {
      "description": "The date or day of the event",
      "title": "Date",
      "type": "string"
    },
    "participants": {
      "description": "List of attendee names",
      "items": {
        "type": "string"
      },
      "title": "Participants",
      "type": "array"
    }
  },
  "required": [
    "name",
    "date",
    "participants"
  ],
  "title": "CalendarEvent",
  "type": "object"
}
```

**How to read it.** The outer dict has four keys:

- `'title': 'CalendarEvent'` is the class name.
- `'type': 'object'` says the whole thing is a group of named fields. In Python terms, a dict.
- `'required': ['name', 'date', 'participants']` lists the fields that must be filled in. All three are listed because none of them has a default value.
- `'properties'` holds one entry per field. Each entry is a small dict of its own, explained next.

Because `schema` is a dict, square brackets pick out one part of it. `schema["properties"]` is the dict of fields, and `["name"]` picks out the `name` field:

```python
print(schema["properties"]["name"])
```

Output:

```
{'description': 'The name or title of the event', 'title': 'Name', 'type': 'string'}
```

Each of these three parts comes from the class definition:

- `'type': 'string'` comes from `name: str`. It is the data type.
- `'description': 'The name or title of the event'` comes from `Field(description="...")`. You typed this sentence into the class yourself. It is an instruction about the field, not a value.
- `'title': 'Name'` is made automatically by Pydantic from the field's name, `name`.

**Why is `participants` an "array"?** It means a list of strings, one string per name. The schema uses JSON's words for types, not Python's, because it is read by OpenAI's server, not by Python. In JSON, a list is called an *array*. So `participants: list[str]` in the class becomes two keys:

- `'type': 'array'` means "this field is a list". It can hold any number of names: `["Alice", "Bob"]`, or just `["Carol"]`.
- `'items': {'type': 'string'}` means "every item in that list must be a string".

This is how each Python type is named in a schema:

- Python `str` → `"string"`
- Python `int` → `"integer"`
- Python `float` → `"number"`
- Python `bool` → `"boolean"`
- Python `list` → `"array"`
- Python `dict`, or a whole class like `CalendarEvent` → `"object"` (which is why the top level of the schema says `'type': 'object'`)

These key names (`type`, `properties`, `items`, `required`) are not Pydantic's own. They come from a public standard called **JSON Schema**, which is why OpenAI's server can read this dict when the SDK sends it.

> **Step 1 in one line:** everything in `schema` comes from the class definition, and nothing comes from an event, because no event exists yet.

---

**Step 2: Creating an event is when real values first appear**

- **What it is:** calling the class like a function, `CalendarEvent(...)`, creates an **instance**: one specific event with real values filled in.
- **The key point:** this is the first time "Science Fair", "Friday", "Alice" and "Bob" appear anywhere.

```python
event = CalendarEvent(name="Science Fair", date="Friday", participants=["Alice", "Bob"])
```

This prints nothing. It stores the new event in the variable `event`.

---

**Step 3: `model_dump()` gives the values of one event**

- **What it is:** `model_dump()` is a Pydantic method that you call on an **instance**, `event`, not on the class. It returns a Python `dict` of that event's values: each field name paired with its value.
- **The key point:** it is the opposite of Step 1. It contains **only values**: no types, no titles, no descriptions.

```python
print(event.model_dump())
```

Output:

```
{'name': 'Science Fair', 'date': 'Friday', 'participants': ['Alice', 'Bob']}
```

Compare the `name` field from the two methods:

- **`schema["properties"]["name"]`** (Step 1, from the class) is `{'description': 'The name or title of the event', 'title': 'Name', 'type': 'string'}`. It describes the field.
- **`event.model_dump()["name"]`** (Step 3, from the event) is `'Science Fair'`. It is the value stored in the field.

---

**Step 4: Two events have different values but the same schema**

- **The key point:** each event has its own values, but they all share one schema, because the schema belongs to the class.

Create a second event and look at its values:

```python
other = CalendarEvent(name="Book Club", date="Monday", participants=["Carol"])
print(other.model_dump())
```

Output:

```
{'name': 'Book Club', 'date': 'Monday', 'participants': ['Carol']}
```

Now compare each event's schema with `schema` from Step 1, which was made before any event existed. `==` checks whether two dicts have exactly the same contents:

```python
print(event.model_json_schema() == schema)
print(other.model_json_schema() == schema)
```

Output:

```
True
True
```

You can call `model_json_schema()` on an event, but it simply asks the event's class, so the answer never changes.

---

**Step 5: `model_dump()` on the class fails**

- **The key point:** `model_dump()` needs an event to read values from, and the class on its own has none.

```python
CalendarEvent.model_dump()
```

Output:

```
TypeError: BaseModel.model_dump() missing 1 required positional argument: 'self'
```

`self` is Python's name for "the specific object this method is working on". Called on the class, there is no such object, so Python stops with this error. Compare Step 1, where `model_json_schema()` worked on the class with no event at all.

---

**Summary**

- **`CalendarEvent.model_json_schema()`** works on the **class**. It describes the **fields** (names, types, descriptions) and contains **no values**.
- **`event.model_dump()`** needs an **instance**. It gives the **values** and contains **no types or descriptions**.

##### Where each one travels

- **The schema goes to the model *before* it writes anything.** You never send it yourself. When you call `parse(response_format=CalendarEvent)`, the SDK creates the schema and puts it in the request. OpenAI's server then uses it as the rules for which tokens the model may write. In the form analogy, this is handing someone the blank form and saying "fill in exactly this".
- **The dump goes wherever *you* send the data, *after* you have it.** This part is your own code:

```python
data = event.model_dump()

# 1. Your own code uses the values.
print("Invite:", ", ".join(data["participants"]))
# -> Invite: Alice, Bob

# 2. A file.
with open("event.json", "w") as f:
    f.write(event.model_dump_json())

# 3. Another service (not run here).
# requests.post("https://example.com/events", json=data)
```

##### Full reference, one call at a time

This part goes through the eight names in this notebook that involve JSON, one at a time. Every entry has the same layout:

- **What it is**: its name, and what it does in one sentence.
- **What goes in, and what comes out.**
- **Which library it belongs to.**
- **Where it appears in this notebook.**
- **The key point**: the one thing to remember.
- **Example**: small code snippets, each followed by its output.

##### Before the examples: meet `event`

Almost every example uses the same event, so let's look at it first. In this notebook, Section 5 gets `event` from `message.parsed`. Here we create an identical one by hand:

```python
event = CalendarEvent(name="Science Fair", date="Friday", participants=["Alice", "Bob"])
print(event)
```

Output:

```
name='Science Fair' date='Friday' participants=['Alice', 'Bob']
```

`print` shows the field values. To also see which class the object belongs to, use `repr()` or `type()`:

```python
print(repr(event))
print(type(event))
```

Output:

```
CalendarEvent(name='Science Fair', date='Friday', participants=['Alice', 'Bob'])
<class '__main__.CalendarEvent'>
```

So `event` is an **object of your class `CalendarEvent`**. It is not a dict. Its structure looks like this:

```
event                  CalendarEvent (your class)
  ├── .name            'Science Fair'        str
  ├── .date            'Friday'              str
  └── .participants    ['Alice', 'Bob']      list of str
        ├── [0]        'Alice'               str
        └── [1]        'Bob'                 str
```

You reach each field with a **dot**:

```python
print(event.name)
print(event.participants)
```

Output:

```
Science Fair
['Alice', 'Bob']
```

Square brackets do **not** work on an object. They only work on dicts and lists:

```python
event["name"]
```

Output:

```
TypeError: 'CalendarEvent' object is not subscriptable
```

##### The three forms your data takes

The same Science Fair data shows up in three different forms in this notebook. Each form is reached in a different way:

- **An object** (a `CalendarEvent`): `CalendarEvent(name='Science Fair', ...)`. You reach its fields with dots: `event.name`.
- **A dict**: `{'name': 'Science Fair', ...}`. You reach its values with square brackets: `d["name"]`. Printed dicts use single quotes.
- **JSON text** (a `str`): `'{"name":"Science Fair", ...}'`. It is just characters, so you can't reach any field until you convert it. JSON always uses double quotes.

Almost every entry below converts data from one form to another:

- **Object → dict:** `event.model_dump()` (entry 2)
- **Object → JSON text:** `event.model_dump_json()` (entry 3)
- **JSON text → dict:** `json.loads()` (entry 4)
- **Dict → JSON text:** `json.dumps()` (entry 5)
- **JSON text → object:** what the SDK does to create `message.parsed` (entry 7)

The other three are different:

- `CalendarEvent.model_json_schema()` (entry 1) describes the *class*, not any data.
- `message.content` (entry 6) is not a conversion. It's where the JSON-text form arrives from the model.
- `response.json()` (entry 8) happens one layer lower, inside the SDK, before any of the others.

---

##### 1. `CalendarEvent.model_json_schema()`

- **What it is:** a Pydantic method that you call on the **class**. It returns a description of the class's fields.
- **What goes in:** the class `CalendarEvent` itself. No event is needed.
- **What comes out:** a Python `dict` describing the *shape*: each field's name, its type and its description. It contains no values.
- **Library:** Pydantic. Every class that inherits from `BaseModel` has this method.
- **Where it appears in this notebook:** Section 2 prints it so you can see it. In normal use you never call it yourself. When you call `parse(response_format=CalendarEvent)`, the SDK calls it for you and sends the result to OpenAI. OpenAI's server uses it to constrain decoding: the model can only write tokens that fit this shape.
- **The key point:** the schema is the shape without the values. "Science Fair" can't be in it.

**Example**

Get the schema. It is an ordinary dict:

```python
schema = CalendarEvent.model_json_schema()
print(type(schema))
```

Output:

```
<class 'dict'>
```

Which fields must be filled in:

```python
print(schema["required"])
```

Output:

```
['name', 'date', 'participants']
```

What the `participants` field must look like:

```python
print(schema["properties"]["participants"])
```

Output:

```
{'description': 'List of attendee names', 'items': {'type': 'string'}, 'title': 'Participants', 'type': 'array'}
```

Read this as a rule: `'type': 'array'` means "a list", and `'items': {'type': 'string'}` means "every item in the list is a string". Together that's "a list of strings", one string per name, which is exactly `list[str]` in the class. "Array" is simply JSON's word for a list (see Step 1 above for how every Python type is named). Notice the rule mentions no names like "Alice" or "Bob".

---

##### 2. `event.model_dump()`

- **What it is:** a Pydantic method that you call on an **instance**. It copies the event's values into a plain Python dict.
- **What goes in:** an instance, such as `event`.
- **What comes out:** a Python `dict` pairing each field name with its value. The values stay ordinary Python objects. For example, `participants` is still a real Python list, not text.
- **Library:** Pydantic.
- **Where it appears in this notebook:** Section 5 calls `event.model_dump()` and prints `{'name': 'Science Fair', ...}`. Section 6 calls `completion.model_dump()` on the whole response object.
- **The key point:** object → dict. You switch from reaching fields with dots (`event.name`) to square brackets (`d["name"]`). You'd do this because many tools, such as `json.dumps`, pandas and databases, accept a dict but have never heard of your class.

**Example**

Convert the event into a dict:

```python
d = event.model_dump()
print(d)
print(type(d))
```

Output:

```
{'name': 'Science Fair', 'date': 'Friday', 'participants': ['Alice', 'Bob']}
<class 'dict'>
```

Compare this with `print(event)` above. The values are the same, but now they sit inside a dict: curly braces, and each field name in quotes followed by a colon.

The same value, reached two ways: a dot on the object, square brackets on the dict:

```python
print(event.name)
print(d["name"])
```

Output:

```
Science Fair
Science Fair
```

Dots don't work on the dict:

```python
d.name
```

Output:

```
AttributeError: 'dict' object has no attribute 'name'
```

"The values stay Python objects": `participants` inside the dict is still a real list, so `[0]` gives the first name:

```python
print(type(d["participants"]))
print(d["participants"][0])
```

Output:

```
<class 'list'>
Alice
```

The dict is a separate copy. Changing it does not change `event`:

```python
d["participants"].append("Carol")
print(d["participants"])
print(event.participants)
```

Output:

```
['Alice', 'Bob', 'Carol']
['Alice', 'Bob']
```

---

##### 3. `event.model_dump_json()`

- **What it is:** a Pydantic method that you call on an **instance**. It turns the event directly into JSON text.
- **What goes in:** an instance, such as `event`.
- **What comes out:** one Python `str` containing JSON. "Skips the dict step" means Pydantic writes the text straight from the object, instead of making a dict first and then converting that.
- **Library:** Pydantic.
- **Where it appears in this notebook:** it isn't used here. It gives roughly the same result as `json.dumps(event.model_dump())`, which is entry 2 followed by entry 5.
- **The key point:** object → text. Once the data is text, it's just characters: no dots and no keys. Use it when data leaves Python, for example when writing a `.json` file or sending it over the network.

**Example**

Convert the event into JSON text:

```python
s = event.model_dump_json()
print(s)
print(type(s))
```

Output:

```
{"name":"Science Fair","date":"Friday","participants":["Alice","Bob"]}
<class 'str'>
```

It looks like the dict from entry 2, but it isn't one. The double quotes show it's JSON text, and `type` confirms it's a `str`.

Because it's text, `[0]` gives the first **character**, and `len` counts characters, not fields:

```python
print(s[0])
print(len(s))
```

Output:

```
{
70
```

You can't look up a field by name any more:

```python
s["name"]
```

Output:

```
TypeError: string indices must be integers, not 'str'
```

The "roughly the same" version, entry 2 then entry 5, gives the same content. The only difference is that `json.dumps` puts a space after each `:` and `,`:

```python
print(json.dumps(event.model_dump()))
```

Output:

```
{"name": "Science Fair", "date": "Friday", "participants": ["Alice", "Bob"]}
```

---

##### 4. `json.loads(s)`

- **What it is:** a function from Python's built-in `json` module. The name means "load string". It reads JSON text and builds the matching Python value.
- **What goes in:** a `str` of JSON text.
- **What comes out:** a `dict` if the text starts with `{`, or a `list` if it starts with `[`.
- **Library:** the standard library's `json` module, which comes with Python. It knows nothing about Pydantic or your class.
- **Where it appears in this notebook:** it isn't called here. You'd need it if you used `create()` instead of `parse()`. `create()` gives you only `message.content`, which is text, so you'd call `json.loads` yourself and then build a `CalendarEvent` from the result. That's "the manual path".
- **The key point:** text → dict. It only reads the text. It checks nothing against `CalendarEvent`.

**Example**

This is the same text that `message.content` holds (entry 6):

```python
raw = '{"name":"Science Fair","date":"Friday","participants":["Alice","Bob"]}'
data = json.loads(raw)
print(data)
print(type(data))
```

Output:

```
{'name': 'Science Fair', 'date': 'Friday', 'participants': ['Alice', 'Bob']}
<class 'dict'>
```

The quotes changed from double to single: it's a Python dict now, so square brackets work but dots don't:

```python
print(data["name"])
```

Output:

```
Science Fair
```

```python
data.name
```

Output:

```
AttributeError: 'dict' object has no attribute 'name'
```

The second manual step turns the dict into your class. `**data` unpacks the dict into named arguments, so this line is the same as writing `CalendarEvent(name="Science Fair", date="Friday", participants=["Alice", "Bob"])`:

```python
event_2 = CalendarEvent(**data)
print(repr(event_2))
```

Output:

```
CalendarEvent(name='Science Fair', date='Friday', participants=['Alice', 'Bob'])
```

If the text is a JSON list, you get a Python list back:

```python
print(json.loads('["Alice", "Bob"]'))
```

Output:

```
['Alice', 'Bob']
```

`json.loads` checks nothing. This text is missing `date` and `participants`, but it reads it without complaint:

```python
half = json.loads('{"name": "Science Fair"}')
print(half)
```

Output:

```
{'name': 'Science Fair'}
```

The problem only shows up at the next step, when `CalendarEvent` checks the fields:

```python
CalendarEvent(**half)
```

Output (shortened):

```
ValidationError: 2 validation errors for CalendarEvent
date
  Field required [type=missing, input_value={'name': 'Science Fair'}, input_type=dict]
participants
  Field required [type=missing, input_value={'name': 'Science Fair'}, input_type=dict]
```

---

##### 5. `json.dumps(obj)`

- **What it is:** a function from Python's built-in `json` module. The name means "dump string". It is the reverse of `json.loads`: it turns a Python value into JSON text.
- **What goes in:** a `dict` or `list` made of plain Python values: `str`, `int`, `float`, `bool`, `None`, and more lists and dicts.
- **What comes out:** a `str` of JSON text.
- **Library:** the standard library's `json` module.
- **Where it appears in this notebook:** Sections 2 and 6 use `json.dumps(..., indent=2)` only to print a dict neatly, one item per line. Nothing is sent anywhere.
- **The key point:** dict → text. It does not understand your class.

**Example**

A plain dict, printed directly and then through `json.dumps`:

```python
d = {"name": "Science Fair", "participants": ["Alice", "Bob"]}
print(d)
print(json.dumps(d))
```

Output:

```
{'name': 'Science Fair', 'participants': ['Alice', 'Bob']}
{"name": "Science Fair", "participants": ["Alice", "Bob"]}
```

The content is the same. The first line is Python's way of showing a dict, with single quotes. The second is JSON text, with double quotes, and it is a `str`:

```python
print(type(json.dumps(d)))
```

Output:

```
<class 'str'>
```

`indent=2` spreads the text over several lines. This is the pretty-printing that Sections 2 and 6 use:

```python
print(json.dumps(d, indent=2))
```

Output:

```
{
  "name": "Science Fair",
  "participants": [
    "Alice",
    "Bob"
  ]
}
```

It only understands plain Python values. Your class is not one of them:

```python
json.dumps(event)
```

Output:

```
TypeError: Object of type CalendarEvent is not JSON serializable
```

The fix is to convert the event into a dict first with `model_dump()` (entry 2). Or use `model_dump_json()` (entry 3), which does both steps at once:

```python
print(json.dumps(event.model_dump()))
```

Output:

```
{"name": "Science Fair", "date": "Friday", "participants": ["Alice", "Bob"]}
```

---

##### 6. `message.content`

- **What it is:** an **attribute** of the message the SDK gives back: a stored value, not a function, which is why there are no brackets `()`. It holds the model's reply exactly as the model wrote it.
- **What it holds:** a `str`. With structured outputs, that string is JSON text shaped like `CalendarEvent`.
- **Library:** the OpenAI SDK. It is a field of the message class, `ParsedChatCompletionMessage` (see the tree in Section 4).
- **Where it appears in this notebook:** Section 4 prints it.
- **The key point:** this is the **JSON-text form** of the data, the same kind of string as `raw` in entry 4 and `s` in entry 3.

**Example**

`message` comes from the response, as in Section 4:

```python
message = completion.choices[0].message
print(message.content)
print(type(message.content))
```

Output:

```
{"name":"Science Fair","date":"Friday","participants":["Alice","Bob"]}
<class 'str'>
```

It is text, so slicing gives characters. `[:8]` takes the first 8:

```python
print(message.content[:8])
```

Output:

```
{"name":
```

With `create()`, this string is all you get. You would carry on with `json.loads` (entry 4) to get a dict, then `CalendarEvent(**data)` to get an object.

---

##### 7. `message.parsed`

- **What it is:** an **attribute** that the SDK adds to the message when you call `parse()`. It holds a ready-made `CalendarEvent` object built from `message.content`.
- **What it holds:** an instance of your class, `CalendarEvent`: the same kind of thing as `event`.
- **Library:** the OpenAI SDK, and only when you use `parse()`. A message from `create()` has no `parsed`.
- **Where it appears in this notebook:** Section 4 prints it, and Section 5 stores it as `event`. This is the payoff of using `parse()`.
- **The key point:** this is the **object form** of the data. The SDK did the text → object conversion for you.

**Example**

It is a `CalendarEvent`, exactly like `event`:

```python
print(repr(message.parsed))
print(type(message.parsed))
```

Output:

```
CalendarEvent(name='Science Fair', date='Friday', participants=['Alice', 'Bob'])
<class '__main__.CalendarEvent'>
```

So dots work straight away, with no conversion:

```python
print(message.parsed.name)
print(message.parsed.participants[1])
```

Output:

```
Science Fair
Bob
```

What the SDK did for you: `model_validate_json` is a Pydantic method on the class. It reads JSON text and builds a checked object in one step, which combines the two manual steps from entry 4 (`json.loads`, then `CalendarEvent(**data)`). `==` on two Pydantic objects is `True` when they are the same class with the same values:

```python
rebuilt = CalendarEvent.model_validate_json(message.content)
print(rebuilt == message.parsed)
```

Output:

```
True
```

---

##### 8. `response.json()`

- **What it is:** a method on an **HTTP response** object. It reads the body of the response, the raw bytes that came over the network, and turns it into a dict.
- **What goes in:** the body of an HTTP response.
- **What comes out:** a `dict` (or a `list`, if the body is a JSON list).
- **Library:** an HTTP client library such as `requests` or `httpx`, *a different library entirely*. It has nothing to do with Pydantic. The OpenAI SDK installed here (v3.14.1) uses `httpx2`, the successor to `httpx` from the same author.
- **Where it appears in this notebook:** never in your code. It runs **inside** the SDK. When OpenAI's server replies, the SDK calls `response.json()` to turn the raw body into a dict. It then builds the typed response object (the `completion` you get back) from that dict. `Supplement_1-basic.ipynb`, Section 4, traces that exact step.
- **The key point:** bytes from the network → dict. It works one layer below everything else in this list. If you've seen `response.json()` elsewhere and filed it with the others, separate it now. The rule is "not in your code", not "never happens".

**Example** (this runs offline, because we build a fake HTTP response by hand)

`httpx2.Response(200, content=...)` creates a response with status code 200 ("OK") and a small JSON body. The `b` before the quotes makes it `bytes`, the raw form data takes on the network:

```python
import httpx2

response = httpx2.Response(200, content=b'{"id": "chatcmpl-123", "object": "chat.completion"}')
print(response.content)
print(type(response.content))
```

Output:

```
b'{"id": "chatcmpl-123", "object": "chat.completion"}'
<class 'bytes'>
```

`response.json()` turns those bytes into a dict:

```python
data = response.json()
print(data)
print(type(data))
```

Output:

```
{'id': 'chatcmpl-123', 'object': 'chat.completion'}
<class 'dict'>
```

It gives the same result as `json.loads` (entry 4) on the body's text. `response.text` is the body decoded from bytes into a `str`:

```python
print(data == json.loads(response.text))
```

Output:

```
True
```

So `response.json()` is roughly `json.loads(response.text)`, plus handling of the text encoding.

---

##### Why Section 6 also calls `.model_dump()`

A neat reinforcement: in Section 6 we call `completion.model_dump()` on the **response object**, not on our own event. That works because the OpenAI SDK's own response classes (`ChatCompletion`, `ParsedChatCompletion`, …) are *themselves* Pydantic `BaseModel` subclasses. Same method, same direction — instance to dict — just applied to a class the SDK authored instead of one you wrote.

##### The one-sentence version

> `model_json_schema()` sends a **shape** outward so the model cannot produce malformed output; `model_dump()` / `model_dump_json()` take a **populated object** and convert it to a dict or string for your own use; `json.loads()` is the generic, non-Pydantic way to turn a JSON string into a dict; `parse()` does the Pydantic version of that step for you, going straight from JSON text to a checked object with `model_validate_json()`.